# Notebook 01: Tokenization & Vocabularies

## Why This Matters
Every LLM interaction starts here. Before a model sees a single weight, your text has been shredded into tokens — integer IDs drawn from a fixed vocabulary. The choices made at this stage (vocabulary size, algorithm, byte-level vs. character-level fallbacks) ripple through everything downstream: memory footprint, context length, cross-lingual capability, and inference cost.

This notebook builds tokenizers from scratch and then dissects the production tokenizers used by real models, so you understand exactly what happens before the first embedding lookup.

## Structure
1. What is a token? Characters, words, subwords, bytes
2. Byte-Pair Encoding (BPE) — built from scratch
3. WordPiece — BERT's algorithm
4. SentencePiece — language-agnostic tokenization
5. Production tokenizers: GPT-2/GPT-4 (tiktoken) vs. LLaMA (SentencePiece BPE)
6. Vocabulary size tradeoffs: fertility, OOV rate, embedding cost
7. Tokenization pathologies: numbers, code, non-English text

## What You'll Learn
- How BPE merges work and why the merge order matters
- Why "tokenization is not reversible" is a real problem for arithmetic
- How vocabulary size affects model size, training cost, and multilingual capability
- Why the same string tokenizes differently across GPT-4, LLaMA, and Mistral

## Background

### What is tokenization?

When you type a message to an LLM, the model never actually reads your words. It reads numbers. Tokenization is the process that converts your text into a sequence of integers — and a tokenizer is the lookup table that defines that mapping.

This seems like a mundane preprocessing detail, but it shapes nearly everything about how a model behaves. How long a sentence "is" to the model depends on the tokenizer. Whether a model can do arithmetic depends heavily on how numbers tokenize. Why GPT-4 is better at English than Thai — despite being exposed to both — comes partly down to tokenization.

### Why subword tokenization?

The community tried simpler approaches first and ran into hard limits:

**Character-level models** (early 2010s) kept the vocabulary tiny (~100 characters) but forced the model to learn spelling, words, and grammar from scratch. Sequences became very long, making training expensive and context windows tiny in terms of meaningful content.

**Word-level models** (most NLP before 2018) kept sequences short but hit the out-of-vocabulary problem: any word not seen in training — typos, names, technical jargon, new words — becomes `[UNK]`, an unknown token the model can't reason about. Vocabulary tables also grew enormous (~100k+ words) when handling multiple languages.

**Subword tokenization** (Sennrich et al., 2016) solved both problems by splitting rare words into smaller pieces while keeping common words whole. The word `"unhappiness"` might tokenize to `["un", "happiness"]` — the model can handle both pieces even if it never saw the full word. No unknown tokens, vocabulary stays manageable (32k–200k), sequences stay reasonably short.

### The three dominant algorithms

**BPE (Byte-Pair Encoding)** was originally a 1994 data compression algorithm. Sennrich adapted it for NLP in 2016, and it's now the foundation for GPT-2, GPT-3, GPT-4, RoBERTa, and most OpenAI models. The core idea: iteratively merge the most frequent pair of symbols in your training corpus until you hit your vocabulary size budget.

**WordPiece** (Google, 2012; used in BERT) uses the same iterative merging structure but picks merges that maximize the likelihood of the training corpus rather than raw frequency. It tends to produce more linguistically meaningful subword units. You can spot WordPiece tokens by the `##` prefix on continuation pieces.

**SentencePiece** (Kudo & Richardson, 2018) addresses a fundamental assumption the others make: that spaces delimit words. Japanese, Chinese, Thai, and Arabic don't use spaces. SentencePiece treats the raw byte stream as input and learns a vocabulary directly, representing spaces as a special `▁` symbol. LLaMA, Mistral, Gemma, and most open-weight models use SentencePiece BPE.

### Why this notebook matters for the rest of the series

Every inference optimization in this series — KV cache sizing, context length, serving cost — is measured in **tokens**, not words or characters. Understanding what a token is, and why the same idea tokenizes differently across models, gives you the foundation to reason about everything that follows.

In [ ]:
# Self-contained install — run once
# %pip install -q tiktoken sentencepiece tokenizers transformers

In [ ]:
import re
import json
import collections
from typing import Dict, List, Tuple

import tiktoken
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import AutoTokenizer

print("Imports OK")

## 1. What Is a Token?

A tokenizer maps a string to a sequence of integers. The mapping is defined by a **vocabulary** — a fixed lookup table of `{token_string: integer_id}`.

Four granularities exist:

```
Character-level:  "hello" → [h, e, l, l, o]         vocab ~100s
Word-level:       "hello" → [hello]                  vocab ~100k+, OOV problem
Subword-level:    "unhappiness" → [un, happiness]    vocab ~32k–100k  ← modern standard
Byte-level:       "hello" → [104, 101, 108, 108, 111] vocab exactly 256, no OOV ever
```

Modern LLMs use **byte-level BPE** (GPT family) or **SentencePiece BPE** (LLaMA family) — subword algorithms that fall back to bytes, eliminating out-of-vocabulary tokens entirely.

## 2. Byte-Pair Encoding (BPE) — From Scratch

BPE was originally a data compression algorithm (1994). Sennrich et al. (2016) adapted it for NLP.

**Algorithm:**
```
1. Start: vocabulary = all characters in corpus
2. Count all adjacent symbol pairs in the corpus
3. Merge the most frequent pair into a new symbol
4. Update the corpus representation
5. Repeat steps 2–4 for N merge operations
```

The merge table is the tokenizer. At inference, you apply the same merges in the same order.

In [ ]:
def get_vocab(corpus: List[str]) -> Dict[Tuple[str, ...], int]:
    """Convert corpus to character-split words with end-of-word marker."""
    vocab = collections.defaultdict(int)
    for word in corpus:
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1
    return dict(vocab)


def get_pairs(vocab: Dict[Tuple[str, ...], int]) -> Dict[Tuple[str, str], int]:
    """Count all adjacent symbol pairs across the vocabulary."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return dict(pairs)


def merge_vocab(pair: Tuple[str, str], vocab: Dict[Tuple[str, ...], int]) -> Dict[Tuple[str, ...], int]:
    """Merge all occurrences of `pair` in vocabulary."""
    merged = {}
    bigram = pair[0] + pair[1]
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        merged[tuple(new_word)] = freq
    return merged


def train_bpe(corpus: List[str], num_merges: int) -> List[Tuple[str, str]]:
    """Train BPE and return the ordered merge table."""
    vocab = get_vocab(corpus)
    merges = []

    print(f"Initial vocab (first 10 entries):")
    for k, v in list(vocab.items())[:10]:
        print(f"  {k}  freq={v}")
    print()

    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
        print(f"Merge {i+1:3d}: {best[0]!r} + {best[1]!r} → {best[0]+best[1]!r}  (freq={pairs[best]})")

    return merges


# Small corpus — deliberately repetitive so merges are easy to trace
corpus = (
    "low lower lowest new newer newest wide wider widest "
    "low lower lowest new newer newest wide wider widest "
    "low lower lowest new newer"
).split()

merges = train_bpe(corpus, num_merges=15)

In [ ]:
def apply_bpe(word: str, merges: List[Tuple[str, str]]) -> List[str]:
    """Tokenize a single word by replaying the merge table."""
    tokens = list(word) + ['</w>']
    for pair in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


test_words = ["low", "lower", "lowest", "newest", "wider", "unknown"]
print("BPE tokenization using learned merges:")
print("-" * 40)
for w in test_words:
    tokens = apply_bpe(w, merges)
    print(f"  {w:12s} → {tokens}")

print()
print("Note: 'unknown' is handled via character fallback — no OOV, but inefficient.")

## 3. WordPiece — BERT's Algorithm

WordPiece (Schuster & Nakamura, 2012; used in BERT) looks similar to BPE but the merge criterion is different:

```
BPE:       merge pair with highest raw frequency
WordPiece: merge pair that maximizes likelihood of the corpus
           score(A, B) = freq(AB) / (freq(A) × freq(B))
```

WordPiece prefers merges that are **surprising** — pairs that co-occur more than expected by chance. This tends to produce more linguistically meaningful subwords.

BERT also uses a `##` prefix convention: tokens that appear mid-word are prefixed with `##` to distinguish them from word-initial occurrences.

```
"playing" → ["play", "##ing"]
"unplayable" → ["un", "##play", "##able"]
```

In [ ]:
# Use HuggingFace tokenizers to demonstrate WordPiece (BERT)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

examples = [
    "tokenization is fundamental",
    "unhappiness and disappointment",
    "cryptocurrency blockchain",
    "supercalifragilisticexpialidocious",
    "42 + 137 = 179",
]

print("BERT WordPiece tokenization:")
print("-" * 50)
for text in examples:
    tokens = bert_tokenizer.tokenize(text)
    ids = bert_tokenizer.encode(text, add_special_tokens=False)
    print(f"Input:  {text!r}")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids}")
    print()

## 4. SentencePiece — Language-Agnostic Tokenization

BPE and WordPiece assume whitespace-delimited words. This breaks for Japanese, Chinese, Thai, and Arabic — languages with no spaces.

**SentencePiece** (Kudo & Richardson, 2018) treats the raw byte stream as input, with no pre-tokenization. It learns vocabulary directly from the character sequence, representing spaces as a special symbol `▁` (U+2581).

```
"Hello world" → [▁Hello, ▁world]    (spaces become part of the token)
"日本語テスト" → [▁日本, 語, テ, スト]  (no pre-tokenization needed)
```

LLaMA, Mistral, Gemma, and most open-weight models use SentencePiece BPE. GPT models use tiktoken BPE.

In [ ]:
# LLaMA 2 tokenizer (SentencePiece BPE)
# Uses the same tokenizer as Mistral 7B and many other open models
llama_tokenizer = AutoTokenizer.from_pretrained("huggyllama/llama-7b")

print("LLaMA SentencePiece tokenization:")
print("-" * 50)
for text in examples:
    tokens = llama_tokenizer.tokenize(text)
    ids = llama_tokenizer.encode(text, add_special_tokens=False)
    print(f"Input:  {text!r}")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids}")
    print()

## 5. Production Tokenizers: tiktoken vs. SentencePiece

tiktoken is OpenAI's tokenizer library — fast, written in Rust, used for GPT-2 through GPT-4o.

Key encodings:
```
gpt2          → 50,257 tokens  (original GPT-2)
cl100k_base   → 100,277 tokens (GPT-3.5, GPT-4)
o200k_base    → 200,019 tokens (GPT-4o)
```

Larger vocabulary = fewer tokens per sentence = cheaper inference, but larger embedding tables.

In [ ]:
import tiktoken

gpt2_enc = tiktoken.get_encoding("gpt2")
gpt4_enc = tiktoken.get_encoding("cl100k_base")
gpt4o_enc = tiktoken.get_encoding("o200k_base")

print(f"Vocabulary sizes:")
print(f"  GPT-2 (gpt2):          {gpt2_enc.n_vocab:>8,}")
print(f"  GPT-4 (cl100k_base):   {gpt4_enc.n_vocab:>8,}")
print(f"  GPT-4o (o200k_base):   {gpt4o_enc.n_vocab:>8,}")
print()

comparison_texts = [
    "The transformer architecture revolutionized natural language processing.",
    "2024-01-15 ERROR: ConnectionTimeout at 192.168.1.1:8080",
    "def attention(q, k, v): return softmax(q @ k.T / k.shape[-1]**0.5) @ v",
    "日本語のテキストはどのようにトークン化されますか",
    "1 + 1 = 2, 1234567890",
]

print(f"{'Text':<55} {'GPT-2':>6} {'GPT-4':>6} {'GPT-4o':>7}")
print("-" * 76)
for text in comparison_texts:
    n_gpt2 = len(gpt2_enc.encode(text))
    n_gpt4 = len(gpt4_enc.encode(text))
    n_gpt4o = len(gpt4o_enc.encode(text))
    display = text[:52] + "..." if len(text) > 55 else text
    print(f"{display:<55} {n_gpt2:>6} {n_gpt4:>6} {n_gpt4o:>7}")

In [ ]:
# Decode token IDs to strings — useful for debugging context windows
text = "Flash attention is O(N) in memory rather than O(N²)."
ids = gpt4_enc.encode(text)

print(f"Text: {text!r}")
print(f"\nToken-by-token breakdown (cl100k_base / GPT-4):")
print(f"{'ID':>8}  Token")
print("-" * 30)
for token_id in ids:
    token_str = gpt4_enc.decode([token_id])
    print(f"{token_id:>8}  {token_str!r}")

print(f"\nTotal tokens: {len(ids)}")

## 6. Vocabulary Size Tradeoffs

Vocabulary size is one of the most consequential hyperparameters in LLM design.

### Fertility
**Fertility** = average number of tokens per word. Lower is better for efficiency.

```
vocab_size ↑  →  fertility ↓  →  shorter sequences  →  cheaper inference, more context
vocab_size ↑  →  embedding table ↑  →  more parameters, more memory
```

### The Embedding Cost
Embedding table size = `vocab_size × hidden_dim`

```
GPT-2:   50,257  × 768   =  38.6M parameters
GPT-4:   100,277 × 12288 = 1.23B parameters  (just the embedding table!)
LLaMA-7B: 32,000 × 4096  = 131M parameters
```

The output projection (lm_head) is typically tied to the embedding table — same weights, transposed — so vocabulary choice doubles this cost.

In [ ]:
# Compute fertility across different tokenizers on the same corpus
test_corpus = """
Large language models are trained on vast corpora of text data using self-supervised learning.
The transformer architecture, introduced in 'Attention Is All You Need' (Vaswani et al., 2017),
uses multi-head self-attention to process sequences in parallel rather than sequentially.
Modern inference optimizations include quantization, speculative decoding, and continuous batching.
These techniques reduce latency and increase throughput for production deployments.
""".strip()

word_count = len(test_corpus.split())

tokenizers_to_compare = {
    "GPT-2 (gpt2, 50k vocab)": gpt2_enc,
    "GPT-4 (cl100k, 100k vocab)": gpt4_enc,
    "GPT-4o (o200k, 200k vocab)": gpt4o_enc,
}

print(f"Corpus word count: {word_count}")
print()
print(f"{'Tokenizer':<35} {'Tokens':>7} {'Fertility':>10}")
print("-" * 55)
for name, enc in tokenizers_to_compare.items():
    tokens = enc.encode(test_corpus)
    fertility = len(tokens) / word_count
    print(f"{name:<35} {len(tokens):>7} {fertility:>10.2f}")

In [ ]:
# Embedding table parameter counts for reference architectures
configs = [
    ("GPT-2",       50_257,  768),
    ("GPT-2 XL",    50_257, 1600),
    ("LLaMA-7B",    32_000, 4096),
    ("LLaMA-70B",   32_000, 8192),
    ("Mistral-7B",  32_000, 4096),
    ("GPT-4 (est)", 100_277, 12288),
]

print(f"{'Model':<18} {'Vocab':>8} {'Hidden':>8} {'Embed Params':>15} {'At BF16':>12}")
print("-" * 65)
for name, vocab, hidden in configs:
    params = vocab * hidden
    gb = params * 2 / 1e9  # BF16 = 2 bytes
    print(f"{name:<18} {vocab:>8,} {hidden:>8,} {params:>15,} {gb:>10.2f}GB")

## 7. Tokenization Pathologies

Tokenization creates systematic failure modes that are worth knowing before you debug model behavior.

### The Numbers Problem
LLMs struggle with arithmetic partly because numbers tokenize inconsistently:
```
"100"   → [100]        (one token)
"1000"  → [1000]       (one token)
"10000" → [100, 00]    (two tokens — the boundary splits mid-number!)
```
The model sees `10000` as two symbols, not one number. Addition and subtraction require reasoning across token boundaries.

### The Code Problem
Indentation in Python code tokenizes as repeated spaces — 4 spaces may be one token or four depending on context. This matters for code generation.

### The Multilingual Problem
Vocabularies trained on English-dominated corpora assign fewer tokens to common English words and more tokens to non-English text:
```
"hello"  → 1 token (GPT-4)
"hola"   → 2 tokens
"こんにちは" → 15 tokens  (same meaning — "hello" in Japanese)
```
Non-English users pay more tokens per idea → shorter effective context → worse performance.

In [ ]:
enc = gpt4_enc

# Numbers tokenization
print("=== Numbers ===")
numbers = ["1", "10", "100", "1000", "10000", "100000", "1000000", "1234567890"]
for n in numbers:
    tokens = [enc.decode([t]) for t in enc.encode(n)]
    print(f"  {n:>12} → {tokens}")

print()
print("=== Arithmetic pathology ===")
for expr in ["1+1", "10+10", "100+100", "1000+1000", "9999+1"]:
    tokens = [enc.decode([t]) for t in enc.encode(expr)]
    print(f"  {expr:<12} → {tokens}")

print()
print("=== Multilingual fertility (GPT-4 cl100k) ===")
greetings = [
    ("English",  "Hello, how are you today?"),
    ("Spanish",  "Hola, ¿cómo estás hoy?"),
    ("French",   "Bonjour, comment allez-vous aujourd'hui?"),
    ("Japanese", "こんにちは、今日はお元気ですか？"),
    ("Arabic",   "مرحبا، كيف حالك اليوم؟"),
    ("Chinese",  "你好，你今天好吗？"),
]
for lang, text in greetings:
    n = len(enc.encode(text))
    words = len(text.split())
    print(f"  {lang:<10} {n:>4} tokens  {text!r}")

In [ ]:
# Token boundary visualization — see exactly where the model "sees" splits
import html

def show_token_boundaries(text: str, enc) -> None:
    """Print text with token boundaries marked by | and alternating case."""
    ids = enc.encode(text)
    parts = [enc.decode([t]) for t in ids]
    annotated = "|".join(f"[{p}]" for p in parts)
    print(f"Input ({len(ids)} tokens): {text!r}")
    print(f"Splits: {annotated}")
    print()

examples = [
    "The attention mechanism computes Q, K, V matrices.",
    "SentencePiece tokenizes without pre-tokenization.",
    "Error at line 42: IndexError: list index out of range",
    "def __init__(self, hidden_size=768, num_heads=12):",
    "The model achieved 94.3% accuracy on the benchmark.",
]

for ex in examples:
    show_token_boundaries(ex, gpt4_enc)

## 8. Training Your Own BPE Tokenizer with HuggingFace

For production use, the HuggingFace `tokenizers` library (Rust-backed) is the right tool — handles special tokens, normalization, padding, truncation, and serialization.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, processors
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence as NormSequence

# Build a BPE tokenizer from scratch
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

# Normalize: NFD unicode decomposition → lowercase → strip accents
tokenizer.normalizer = NormSequence([NFD(), Lowercase(), StripAccents()])

# Pre-tokenize: split on whitespace and punctuation
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Trainer
trainer = trainers.BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
)

# Small corpus for demonstration
training_corpus = [
    "the transformer architecture uses multi-head self-attention",
    "attention is all you need for sequence to sequence tasks",
    "byte pair encoding builds a vocabulary from character merges",
    "tokenization maps text to integer token identifiers",
    "large language models are trained on token sequences",
    "vocabulary size affects both fertility and embedding table size",
    "the key value cache stores attention computations for efficiency",
    "quantization reduces model precision to decrease memory usage",
    "speculative decoding uses a draft model to accelerate inference",
    "continuous batching improves gpu utilization during inference",
] * 50  # repeat to give BPE enough signal

tokenizer.train_from_iterator(training_corpus, trainer)

print(f"Trained vocabulary size: {tokenizer.get_vocab_size()}")
print()

test = "the transformer uses attention mechanisms for language modeling"
encoding = tokenizer.encode(test)
print(f"Input:  {test!r}")
print(f"Tokens: {encoding.tokens}")
print(f"IDs:    {encoding.ids}")

### WordPiece — from scratch with `tokenizers`

WordPiece uses the same bottom-up merge approach as BPE but picks merges differently:

```
BPE score:       freq(AB)
WordPiece score: freq(AB) / (freq(A) × freq(B))
```

This ratio rewards pairs that are *surprisingly* common — co-occurring more than chance would predict. The result is more linguistically motivated subwords.

Mid-word pieces get a `##` prefix so the tokenizer can reconstruct original whitespace:
```
"tokenization" → ["token", "##ization"]
```

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence as NormSequence

# WordPiece tokenizer — same corpus as BPE above
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

wp_tokenizer.normalizer = NormSequence([NFD(), Lowercase(), StripAccents()])
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

wp_trainer = trainers.WordPieceTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    continuing_subword_prefix="##",  # mid-word pieces get this prefix
)

training_corpus = [
    "the transformer architecture uses multi-head self-attention",
    "attention is all you need for sequence to sequence tasks",
    "byte pair encoding builds a vocabulary from character merges",
    "tokenization maps text to integer token identifiers",
    "large language models are trained on token sequences",
    "vocabulary size affects both fertility and embedding table size",
    "the key value cache stores attention computations for efficiency",
    "quantization reduces model precision to decrease memory usage",
    "speculative decoding uses a draft model to accelerate inference",
    "continuous batching improves gpu utilization during inference",
] * 50

wp_tokenizer.train_from_iterator(training_corpus, wp_trainer)

print(f"WordPiece vocabulary size: {wp_tokenizer.get_vocab_size()}")
print()

test = "the transformer uses attention mechanisms for language modeling"
enc = wp_tokenizer.encode(test)
print(f"Input:  {test!r}")
print(f"Tokens: {enc.tokens}")
print(f"IDs:    {enc.ids}")
print()

# ## prefix shows which pieces are continuations vs. word-starts
continuation_pieces = [t for t in wp_tokenizer.get_vocab() if t.startswith("##")]
print(f"Continuation pieces (##) in vocab: {len(continuation_pieces)}")
print(f"Examples: {sorted(continuation_pieces)[:15]}")

### Unigram — from scratch with `tokenizers`

Unigram works **top-down** — the opposite of BPE and WordPiece:

1. Start with a large over-complete vocabulary (all substrings up to some length)
2. Assign each token a log-probability under a unigram language model
3. For each token, compute how much the corpus log-likelihood drops if you remove it
4. Prune the bottom X% of least-useful tokens
5. Repeat until target vocab size is reached

This means every segmentation has an explicit probability, and the tokenizer chooses the **most probable** segmentation — not a greedy left-to-right one. During training, alternative segmentations can be sampled (subword regularization) as data augmentation.

Used by SentencePiece (T5, ALBERT) when configured in Unigram mode.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence as NormSequence

# Unigram tokenizer
uni_tokenizer = Tokenizer(models.Unigram())

uni_tokenizer.normalizer = NormSequence([NFD(), Lowercase(), StripAccents()])
uni_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

uni_trainer = trainers.UnigramTrainer(
    vocab_size=500,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    unk_token="[UNK]",
)

uni_tokenizer.train_from_iterator(training_corpus, uni_trainer)

print(f"Unigram vocabulary size: {uni_tokenizer.get_vocab_size()}")
print()

test = "the transformer uses attention mechanisms for language modeling"
enc = uni_tokenizer.encode(test)
print(f"Input:  {test!r}")
print(f"Tokens: {enc.tokens}")
print(f"IDs:    {enc.ids}")

## 9. Bridge to Embeddings

Token IDs are integers — the model cannot do math on integers directly. The first thing the transformer does is look each ID up in an **embedding matrix** $W_E$ of shape `[vocab_size, hidden_dim]`.

```
token_id  →  W_E[token_id]  →  vector of shape [hidden_dim]
```

This is just an array index — no computation, just a lookup. The embedding matrix is a learned parameter: each row is a `hidden_dim`-dimensional point in space that the model learns to position so that semantically similar tokens end up nearby.

A full sequence goes through the same lookup in parallel:
```
[101, 2054, 2003, 3086, 102]  →  W_E[[101, 2054, 2003, 3086, 102]]
                               →  tensor of shape [seq_len, hidden_dim]
```

This tensor — one vector per token — is what flows into the first transformer layer. **Everything the model knows about a token at position 0 comes from this single row lookup.** Context only accumulates through the attention layers that follow.

Notebook 02 starts here.

In [ ]:
import numpy as np

# Tiny demonstration: what the embedding lookup looks like mechanically
vocab_size = 100   # toy vocab
hidden_dim = 16    # toy embedding dimension
rng = np.random.default_rng(42)

# W_E is the embedding matrix — one row per token in the vocabulary
W_E = rng.standard_normal((vocab_size, hidden_dim)).astype(np.float32)
print(f"Embedding matrix shape: {W_E.shape}  (vocab_size x hidden_dim)")
print()

# Tokenize a sentence and look up each token
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

sentence = "attention is all you need"
token_ids = enc.encode(sentence)
tokens    = [enc.decode([t]) for t in token_ids]

print(f"Sentence: {sentence!r}")
print(f"Token IDs: {token_ids}")
print(f"Tokens:    {tokens}")
print()

# Clamp IDs to toy vocab size for demo purposes
demo_ids = [t % vocab_size for t in token_ids]

# The embedding lookup — this is literally what nn.Embedding does
embedded = W_E[demo_ids]   # shape: [seq_len, hidden_dim]
print(f"Embedded shape: {embedded.shape}  (seq_len x hidden_dim)")
print()
print("One vector per token (first 6 dims shown):")
for token, vec in zip(tokens, embedded):
    print(f"  {token!r:15s} → {vec[:6].round(3)}")

print()
print("This tensor is the input to transformer layer 0.")
print("The model has seen nothing else — no context, no position, just these lookup vectors.")
print("Positional encodings are added next (Notebook 02).")

In [ ]:
# Side-by-side comparison of all three on the same inputs
test_sentences = [
    "the transformer uses attention mechanisms for language modeling",
    "tokenization maps text to integer identifiers",
    "speculative decoding accelerates inference",
    "unseen vocabulary words like hyperbolic",  # OOV challenge
]

print(f"{'Input':<45} {'Algorithm':<12} {'Tokens'}")
print("-" * 90)
for text in test_sentences:
    for name, tok in [("BPE", tokenizer), ("WordPiece", wp_tokenizer), ("Unigram", uni_tokenizer)]:
        enc = tok.encode(text)
        print(f"{text[:44]:<45} {name:<12} {enc.tokens}")
    print()

## Summary

| Algorithm | Used by | Key idea | Merge criterion |
|-----------|---------|----------|-----------------|
| BPE | GPT family, RoBERTa | Compress frequent pairs | Highest frequency |
| WordPiece | BERT, DistilBERT | Maximize corpus likelihood | freq(AB) / freq(A)×freq(B) |
| SentencePiece | LLaMA, Mistral, Gemma | No pre-tokenization, language-agnostic | BPE or Unigram LM |
| tiktoken | GPT-3.5, GPT-4, GPT-4o | Fast Rust BPE, byte-level fallback | Frequency (byte-level) |

**Key takeaways:**
- Tokenization is lossy: the model never sees your raw text, only integer IDs
- Vocabulary size trades embedding cost against sequence length (fertility)
- Numbers, code, and non-English text tokenize poorly in English-dominant vocabularies
- The merge table is the tokenizer — inference replays the same merges in the same order
- BPE and its variants eliminate OOV tokens entirely via byte-level fallback

**Next:** Notebook 02 builds the transformer architecture that consumes these token IDs — embeddings, attention, MLP blocks, and the residual stream.